# NB5 — Gộp SFT+DPO → GGUF Q4_K_M (TUỲ CHỌN, thưởng điểm)

> Phần lõi của lab = NB0–NB4. Bước này biên dịch llama.cpp lúc chạy (~3–5 phút lần đầu).

**Lỗi của lab cũ:** NB5 chỉ nạp adapter **SFT** rồi gộp, nên file GGUF không có DPO.
Ở đây ta nạp `adapters/dpo` (adapter config trỏ tới `models/sft-merged`), tức là
SFT + DPO, rồi xuất file. Có một bước kiểm tra để chắc chắn adapter DPO thực sự được nạp.

In [1]:
import json
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lab22" / "config.py").exists())
sys.path.insert(0, str(ROOT))

import unsloth  # noqa: F401
import torch

from lab22 import config as C
from lab22 import modeling as MD

assert torch.cuda.is_available()
assert (C.DPO_ADAPTER / "adapter_config.json").exists(), "Run NB3 first"
C.ensure_dirs()

adapter_cfg = json.loads((C.DPO_ADAPTER / "adapter_config.json").read_text())
print(f"DPO adapter base: {adapter_cfg.get('base_model_name_or_path')}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!


DPO adapter base: /content/lab22-runtime/models/sft-merged


## 1. Nạp SFT + DPO ở 16-bit

Gộp vào trọng số 4-bit làm mất độ chính xác, nên nạp 16-bit (`load_in_4bit=False`).
T4 16 GB đủ cho 4B ở fp16 (~8 GB).

In [2]:
model, tokenizer = MD.load_model(C.DPO_ADAPTER, load_in_4bit=False)
n_lora = sum(1 for n, _ in model.named_parameters() if "lora_" in n)
assert n_lora > 0, "DPO LoRA weights not loaded: the GGUF would be SFT-only"
print(f"LoRA tensors loaded: {n_lora}")

SMOKE_PROMPT = "Giải thích ngắn gọn (3 câu) cách thuật toán Bubble sort hoạt động."
SMOKE_TOKENS = 160
hf_answer = MD.generate(model, tokenizer, [SMOKE_PROMPT], max_new_tokens=SMOKE_TOKENS)[0]
# Render the prompt once with the HF template (thinking off) and feed GGUF the same text,
# so the HF vs GGUF comparison differs only by quantization, not by chat formatting.
smoke_text = tokenizer.apply_chat_template(
    [{"role": "user", "content": SMOKE_PROMPT}], tokenize=False, add_generation_prompt=True, **C.CHAT_TEMPLATE_KWARGS
)
stop_tokens = [tokenizer.eos_token] if tokenizer.eos_token else []
print(f"HF (SFT+DPO) answer:\n{hf_answer}")

==((====))==  Unsloth 2026.10.3: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Unsloth 2026.10.3 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


LoRA tensors loaded: 504


Both `max_new_tokens` (=160) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


HF (SFT+DPO) answer:
</tool_call>

</tool_call>

Thuật toán Bubble Sort hoạt động bằng cách so sánh các cặp phần tử liên tiếp trong danh sách và đổi chỗ chúng nếu chúng không ở đúng thứ tự. Quá trình này lặp đi lặp lại cho đến khi không còn có cặp phần tử nào cần được đổi chỗ, nghĩa là danh sách đã được sắp xếp.


## 2. Xuất GGUF Q4_K_M

`save_pretrained_gguf` gộp LoRA vào trọng số 16-bit rồi gọi llama.cpp để lượng tử hoá.
Tên file và thư mục con khác nhau giữa các bản Unsloth, nên ta tìm file bằng glob đệ quy.

In [3]:
import os
export_work = C.GGUF_DIR.parent / "gguf-work"
export_work.mkdir(parents=True, exist_ok=True)
previous_cwd = Path.cwd()
try:
    os.chdir(export_work)
    model.save_pretrained_gguf(str(C.GGUF_DIR), tokenizer, quantization_method="q4_k_m")
finally:
    os.chdir(previous_cwd)
# Optional for the +3 rigor add-on: quantization_method=["q4_k_m", "q5_k_m", "q8_0"]


def find_gguf(pattern: str = "q4_k_m") -> Path:
    hits = [p for p in C.GGUF_DIR.parent.glob(C.GGUF_DIR.name + "*/**/*.gguf") if pattern in p.name.lower()]
    assert hits, f"No *{pattern}*.gguf under {C.REPO_ROOT}/gguf*"
    return max(hits, key=lambda p: p.stat().st_mtime)


gguf_path = find_gguf()
print(f"{gguf_path}  {gguf_path.stat().st_size / 1e9:.2f} GB")
del model
MD.cleanup()

Unsloth: Merging model weights to 16-bit format...
Detected local model directory: /content/lab22-runtime/models/sft-merged


Unsloth: Restored added_tokens_decoder metadata in /content/lab22-runtime/gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

Unsloth: Preparing safetensor model files:  50%|█████     | 1/2 [00:23<00:23, 23.47s/it]

...

Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...


Unsloth: Initial conversion completed! Files: ['/content/lab22-runtime/gguf_gguf/sft-merged.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...


Unsloth: Model files cleanup...


Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/lab22-runtime/gguf_gguf/sft-merged.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model '/content/lab22-runtime/models/sft-merged'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /content/lab22-runtime/gguf_gguf/sft-merged.Q4_K_M.gguf -p "why is the sky blue?"
/content/lab22-runtime/gguf_gguf/sft-merged.Q4_K_M.gguf  2.50 GB


## 3. Kiểm tra nhanh (smoke test) bằng llama-cpp-python (sản phẩm nộp `06-gguf-smoke.png`)

So câu trả lời GGUF với câu trả lời HF ở §1. Lượng tử hoá 4-bit làm câu chữ lệch một
chút, nhưng nội dung và phong cách phải giống nhau. Nếu GGUF trả lời giống hệt
mô hình SFT (NB4) thì adapter DPO đã bị bỏ sót.

In [4]:
from llama_cpp import Llama

llm = Llama(model_path=str(gguf_path), n_ctx=C.MAX_LEN, n_gpu_layers=-1, verbose=False)
resp = llm.create_completion(smoke_text, max_tokens=SMOKE_TOKENS, temperature=0.0, stop=stop_tokens)
gguf_answer = resp["choices"][0]["text"].strip()
print(f"PROMPT: {SMOKE_PROMPT}\n\nGGUF Q4_K_M:\n{gguf_answer}\n\nusage: {resp['usage']}")
llm.close()  # release the llama.cpp GPU buffers before the next notebook
del llm

PROMPT: Giải thích ngắn gọn (3 câu) cách thuật toán Bubble sort hoạt động.

GGUF Q4_K_M:
</tool_call>

</tool_call>

Thuật toán Bubble Sort hoạt động bằng cách so sánh các cặp phần tử liên tiếp trong danh sách và hoán đổi chúng nếu chúng không ở vị trí đúng. Quá trình này lặp đi lặp lại cho đến khi không còn hoán đổi nào có thể thực hiện được, nghĩa là danh sách đã được sắp xếp.

usage: {'prompt_tokens': 25, 'completion_tokens': 72, 'total_tokens': 97}


In [5]:
deploy_meta = {
    "compute_tier": C.COMPUTE_TIER,
    "base_model": C.BASE_MODEL,
    "adapter": str(C.DPO_ADAPTER.relative_to(C.REPO_ROOT)),
    "adapter_base": adapter_cfg.get("base_model_name_or_path"),
    "gguf_path": str(gguf_path),
    "gguf_size_mb": round(gguf_path.stat().st_size / 1e6, 1),
    "quantization": "q4_k_m",
    "smoke_prompt": SMOKE_PROMPT,
    "chat_template_kwargs": C.CHAT_TEMPLATE_KWARGS,
    "max_new_tokens": SMOKE_TOKENS,
    "hf_answer": hf_answer,
    "gguf_answer": gguf_answer,
}
(C.EVAL_DIR / "deploy_meta.json").write_text(json.dumps(deploy_meta, ensure_ascii=False, indent=2))
print("Saved data/eval/deploy_meta.json")

Saved data/eval/deploy_meta.json


## 4. Triển khai (tham khảo, chạy ngoài notebook)

Máy chủ llama.cpp (CPU/GPU, file GGUF):

```bash
llama-server -m gguf/<file>.Q4_K_M.gguf --port 8080 -c 2048
```

vLLM (BigGPU, ≥16 GB): phục vụ mô hình 16-bit có LoRA mà không cần gộp:

```bash
vllm serve models/sft-merged --enable-lora --lora-modules dpo=adapters/dpo \
  --max-model-len 2048 --port 8000
```

Gọi API với `"model": "dpo"`. vLLM giữ tiến trình và GPU, nên chạy ở cửa sổ dòng lệnh riêng.

**Tiếp theo:** NB6 (benchmark) hoặc `make verify`.